In [1]:
import fastf1
import pandas as pd
import numpy as np

# Enable FastF1 cache
fastf1.Cache.enable_cache('D:\\Programming\\Python\\AI\\F1 Fantasy\\cache')

In [2]:
round_number = 16
year = 2026

In [3]:
def get_push_laps(session, threshold=1.01):
    laps = session.laps.copy()
    laps['Session'] = session.name
    laps = laps.pick_accurate().pick_wo_box().pick_track_status('1', how='contains')
    laps = laps[laps['LapTime'].notna()]
    laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()

    push_laps = pd.DataFrame()

    for driver, driver_laps in laps.groupby('Driver'):
        personal_best = driver_laps['LapTimeSeconds'].min()

        if pd.isna(personal_best):
            continue

        # threshold: Push laps should be close together relative to driver's personal best
        driver_push_laps = driver_laps[driver_laps['LapTimeSeconds'] <= personal_best * threshold]

        push_laps = pd.concat([push_laps, driver_push_laps], ignore_index=True)

    return push_laps

In [4]:
fp1 = fastf1.get_session(year, round_number, 'fp1')
fp1.load(telemetry=False)
fp2 = fastf1.get_session(year, round_number, 'fp2')
fp2.load(telemetry=False)
fp3 = fastf1.get_session(year, round_number, 'fp3')
fp3.load(telemetry=False)

core           INFO 	Loading data for Bahrain Grand Prix - Practice 1 [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['1', '3', '5', '6', '10', '11', '12', '14', '16', '18', '23', '27', '30', '31', '41', '43', '44', '55', '63', '77', '81', '87']
core           INFO 	Loading data for Bahrain Grand Prix - Practice 2 [v3.8.3]
req           

In [5]:
import statsmodels.formula.api as smf

# USE FIRST LINE WHEN NORMAL WEEKEND, USE SECOND LINE WHEN SPRINT WEEKEND (THERES ONLY 1 PRACTICE SESSION)
push = pd.concat([get_push_laps(fp1), get_push_laps(fp2), get_push_laps(fp3)], ignore_index=True)
#push = get_push_laps(fp1)
push['evo'] = (push['LapStartTime'].dt.total_seconds() - push['LapStartTime'].dt.total_seconds().min() / 3600.0)

model_push = smf.rlm(
    'LapTimeSeconds ~ 0 + C(Driver) + C(Compound) + C(Session) + evo',
    data=push
).fit()

push_qpace_raw = model_push.params.filter(like='C(Driver')
push_qpace_raw.index = push_qpace_raw.index.str.extract(r'\[(.*)\]')[0].values
push_qpace_raw -= push_qpace_raw.min()

push_qpace_se = model_push.bse.filter(like='C(Driver)')
push_qpace_se.index = push_qpace_se.index.str.extract(r'\[(.*)\]')[0].values

session_best = push['LapTimeSeconds'].min()
push_qpace_raw.sort_values()

VER    0.000000
RUS    0.171723
HAD    0.276153
LEC    0.308782
NOR    0.365025
ANT    0.445030
HAM    0.586202
PIA    0.952089
LAW    1.301566
LIN    1.351770
GAS    1.401209
BOR    1.776348
HUL    1.795996
ALO    1.884411
OCO    1.935444
BEA    2.329513
ALB    2.498213
SAI    2.501997
STR    2.605513
BOT    3.224316
PER    3.757224
COL    5.561794
dtype: float64

In [6]:
# increaseWeightFactor added to give practice pace more equal weight compared to historical qpace
# smaller value will give more weight to practice pace (adjust as needed)
def source_practice_qpace(driver, push_qpace_raw, push_qpace_se, session_best, increaseWeightFactor=.2):
    if driver not in push_qpace_raw.index:
        return 0.0, np.inf
    mu = push_qpace_raw[driver] / session_best
    sigma = (push_qpace_se[driver] / session_best) * increaseWeightFactor
    return mu, sigma

In [58]:
qualiRound = 13
quali = fastf1.get_session(year, qualiRound, 'q')
quali.load(telemetry=False)

core           INFO 	Loading data for Italian Grand Prix - Qualifying [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 

In [59]:
push = get_push_laps(quali).copy()
push['evo'] = (push['LapStartTime'].dt.total_seconds() - push['LapStartTime'].dt.total_seconds().min() / 3600.0)

model_push = smf.rlm(
    'LapTimeSeconds ~ 0 + C(Driver) + C(Compound) + C(Session) + evo',
    data=push
).fit()

qpace_raw = model_push.params.filter(like='C(Driver)')
qpace_raw.index = qpace_raw.index.str.extract(r'\[(.*)\]')[0].values
qpace_raw -= qpace_raw.min()

qpace_se = model_push.bse.filter(like='C(Driver)')
qpace_se.index = qpace_se.index.str.extract(r'\[(.*)\]')[0].values

session_best = push['LapTimeSeconds'].min()

qpace = qpace_raw / push['LapTimeSeconds'].median()
qpace_se_frac = qpace_se / push['LapTimeSeconds'].median()
qpace

ALB    0.023662
ALO    0.034422
ANT    0.001217
BEA    0.009192
BOR    0.007158
BOT    0.025591
COL    0.004932
GAS    0.000033
HAM    0.005845
HUL    0.012382
LAW    0.009239
LEC    0.003017
LIN    0.004635
NOR    0.003617
OCO    0.017959
PER    0.028214
PIA    0.000459
RUS    0.000000
SAI    0.017508
STR    0.036519
TSU    0.017494
VER    0.003127
dtype: float64

In [60]:
round_result = pd.DataFrame({
    'Driver': qpace_raw.index,
    'Team': [push[push.Driver == d].Team.values[0] for d in qpace_raw.index],
    'Round': qualiRound,
    'Year': year,
    'qpace_frac': qpace.values,
    'qpace_se_frac': qpace_se_frac.values,
})
round_result

,Driver,Team,Round,Year,qpace_frac,qpace_se_frac
0,ALB,Williams,13,2026,0.023662,0.003567
1,ALO,Aston Martin,13,2026,0.034422,0.003144
2,ANT,Mercedes,13,2026,0.001217,0.003075
3,BEA,Haas F1 Team,13,2026,0.009192,0.002930
4,BOR,Audi,13,2026,0.007158,0.002927
5,BOT,Cadillac,13,2026,0.025591,0.003145
6,COL,Alpine,13,2026,0.004932,0.002883
7,GAS,Alpine,13,2026,0.000033,0.003019
8,HAM,Ferrari,13,2026,0.005845,0.003020
9,HUL,Audi,13,2026,0.012382,0.003024


In [14]:
quali_archive = pd.read_csv('data/quali_archive.csv')
quali_archive = pd.concat([quali_archive, round_result], ignore_index=True)
quali_archive.to_csv('data/quali_archive.csv', index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'data/quali_archive.csv'

In [7]:
quali_archive = pd.read_csv('data/quali_archive.csv')

In [8]:
def source_quali_form(driver, archive, current_round, half_life=4):
    rows = archive[archive['Driver'] == driver]
    if rows.empty:
        return 0.0, np.inf

    ests, ses, weights = [], [], []
    for round, group in rows.groupby('Round'):
        age = current_round - round
        if age <= 0:
            continue
            
        age_w = 0.5 ** (age / half_life)
        ests.append(group['qpace_frac'].iloc[0])
        ses.append(group['qpace_se_frac'].iloc[0])
        weights.append(age_w)

    if not weights or sum(weights) == 0:
        return 0.0, np.inf

    ests, ses, weights = map(np.array, (ests, ses, weights))
    w = weights / ses**2
    mu = np.sum(w * ests) / np.sum(w)
    sigma = np.sqrt(1.0 / np.sum(w))
    return mu, sigma

In [9]:
def blend_sources(sources, tau):
    # sources: dict name -> (mu, sigma)   sigma = sampling SE, fractional units
    # tau: drift sd added
    names, mu, var = [], [], []
    for name, (m, s) in sources.items():
        if not np.isfinite(m) and np.isfinite(s):
            continue
        names.append(name)
        mu.append(m)
        var.append(s**2 + tau.get(name, 0.0)**2)

    if not names:
        return np.nan, np.inf, {}

    mu, var = np.array(mu), np.array(var)
    w = 1.0 / var
    blended_mu = float(np.sum(w * mu) / w.sum())
    blended_sigma = float(np.sqrt(1.0 / w.sum()))
    return blended_mu, blended_sigma, dict(zip(names, w / w.sum()))

In [10]:
# NEW VERSION
TAU_Q = {'practice': 0.0025, 'quali_form': 0.0035}

def blend_qpace_tau(driver, current_round, push_qpace_raw, push_qpace_se, push_session_best, quali_archive, tau=TAU_Q):
    # practice_mu = source_practice_qpace(driver, push_qpace_raw, push_qpace_se, push_session_best),
    # quali_mu, quali_sigma = source_quali_form(driver, quali_archive, current_round)
    sources = {
        'practice': (source_practice_qpace(driver, push_qpace_raw, push_qpace_se, push_session_best)),
        'quali_form': (source_quali_form(driver, quali_archive, current_round))
    }
    return blend_sources(sources, tau)

q_mu, q_sigma, q_shares = {}, {}, {}
for d in push.Driver:
    q_mu[d], q_sigma[d], q_shares[d] = blend_qpace_tau(d, round_number, push_qpace_raw, push_qpace_se, session_best, quali_archive)

current_qpace = pd.Series(q_mu)
qpace_sigma   = pd.Series(q_sigma)      # now includes drift, so it is wider than before
qpace_shares  = pd.DataFrame(q_shares).T
print(qpace_shares.mean())          # average weight each source carries
print(qpace_shares.round(2).loc[['ANT', 'VER', 'STR']])
current_qpace = pd.concat([current_qpace, qpace_sigma], axis=1, keys=['mu', 'sigma'])
current_qpace.sort_values(by='mu')

practice      0.657979
quali_form    0.342021
dtype: float64
     practice  quali_form
ANT      0.66        0.34
VER      0.66        0.34
STR      0.67        0.33


,mu,sigma
VER,0.001527,0.002169
RUS,0.002320,0.002173
LEC,0.003185,0.002131
NOR,0.003377,0.002179
ANT,0.003405,0.002164
HAD,0.003822,0.002188
HAM,0.005290,0.002168
PIA,0.007903,0.002162
LAW,0.012514,0.002162
LIN,0.012524,0.002164


In [11]:
current_qpace.to_csv("current_qpace.csv")

In [23]:
# OLD VERSION
def blend_qpace(driver, team, current_round, push_qpace_raw, push_qpace_se, push_session_best, quali_archive):
    sources = [
        source_practice_qpace(driver, push_qpace_raw, push_qpace_se, push_session_best),
        source_quali_form(driver, quali_archive, current_round)
    ]

    mu = np.array([s[0] for s in sources])
    sigma = np.array([s[1] for s in sources])
    w = 1.0 / sigma**2
    
    if not np.isfinite(w).any():
        raise ValueError(f"No qpace source available for {driver}")
        
    return np.sum(w * mu) / np.sum(w), np.sqrt(1.0 / np.sum(w))

current_qpace = {}
qpace_sigma = {}
for driver in push.Driver:
    mu, sigma = blend_qpace(
        driver, push[push.Driver == driver].Team.values[0], round_number, push_qpace_raw, push_qpace_se, session_best, quali_archive
    )
    current_qpace[driver] = mu
    qpace_sigma[driver] = sigma

current_qpace = pd.Series(current_qpace)
qpace_sigma = pd.Series(qpace_sigma)

current_qpace = pd.concat([current_qpace, qpace_sigma], axis=1, keys=['mu', 'sigma'])
current_qpace.sort_values(by='mu')

,mu,sigma
LEC,0.001726,0.000748
ANT,0.001940,0.000749
HAM,0.002079,0.000757
NOR,0.003132,0.000770
VER,0.004081,0.000752
RUS,0.004392,0.000753
PIA,0.005425,0.000765
HAD,0.006688,0.000739
LAW,0.011846,0.000731
LIN,0.012381,0.000729


In [87]:
d = 'HAM'
print(source_practice_qpace(d, push_qpace_raw, push_qpace_se, session_best)),
print(source_quali_form(d, quali_archive, 11))

(np.float64(0.0), np.float64(0.001065607220663182))
(np.float64(0.0034322626888673097), np.float64(0.000977023127851868))


In [33]:
# does source_quali_form alone show real variation, independent of the practice source?
qf_results = {d: source_quali_form(d, quali_archive, qualiRound) for d in push.Driver}
qf_sigma = pd.Series({d: v[1] for d, v in qf_results.items()})
qf_sigma.describe()

d:\Programming\Python\AI\F1 Fantasy\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


count    24.000000
mean           inf
std            NaN
min       0.001023
25%       0.001100
50%       0.001139
75%       0.001228
max            inf
dtype: float64

In [37]:
qf_sigma = pd.Series({d: source_quali_form(d, quali_archive, qualiRound)[1] for d in push.Driver})
qf_sigma[~np.isfinite(qf_sigma)]

FOR    inf
VES    inf
dtype: float64

In [38]:
set(push.Driver) - set(quali_archive['Driver'].unique())

{'FOR', 'VES'}

In [42]:
quali_archive[quali_archive['Driver'].isin(['FOR', 'VES'])]

,Driver,Team,Round,Year,qpace_frac,qpace_se_frac
